# SpatialChat — Step-by-Step Tracing Notebook

Traces **every call** through the full agent pipeline, from config to graph output.

### Architecture (post-rewrite)
- **`bind_tools`** for multi-step ReAct loop with proper `ToolMessage` + `tool_call_id` matching
- **Pydantic validation** on tool args via LangChain's `@tool` decorator with `args_schema`
- **PlotStore** keeps base64 out of LLM context (tools return `plot_id` only)
- **MemorySaver checkpointer** for multi-turn conversation (per-invocation state reset by `reset` node)
- **Compact tool summaries** — never full gene expression or large data in state
- **Plots embedded** as `image_url` in final AIMessage for LangGraph Studio rendering
- **Universal h5ad loader** — no per-dataset functions, all datasets load from h5ad files
- **Celltype-aware tools** — plot celltypes spatially, gene expression by celltype
- **Metadata store** — pre-computed gene lists + cell types per dataset for fast lookups and fuzzy gene matching

### LangSmith Tracing
Every LLM call, tool invocation, and graph step is traced in LangSmith.
Set your `LANGSMITH_API_KEY` in `.env` to see traces at https://smith.langchain.com

---
Run cells top-to-bottom. The test query is: **"Sox2 expression in Mouse Brain seqFISH"**

## 0. Setup & Environment

In [ ]:
import os, sys, json, importlib
sys.path.insert(0, os.path.abspath("."))

from dotenv import load_dotenv
load_dotenv()

print("ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("OPENAI_API_KEY set:  ", bool(os.environ.get("OPENAI_API_KEY")))
print("LANGSMITH_API_KEY:   ", bool(os.environ.get("LANGCHAIN_API_KEY") or os.environ.get("LANGSMITH_API_KEY")))
print("LANGSMITH_PROJECT:   ", os.environ.get("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "spatialchat")))

In [ ]:
# Hot-reload helper — run this cell after editing source files
def reload_all():
    mods = [
        'config.settings', 'tools.base', 'tools.dataset_tools',
        'tools.expression_tools', 'tools.stats_tools', 'tools.neighbor_tools',
        'data.loaders', 'data.metadata_store', 'agents.state', 'agents.supervisor',
        'agents.sub_agents', 'graph',
    ]
    for m in mods:
        if m in sys.modules:
            importlib.reload(sys.modules[m])
    import graph as _g
    _g._compiled = None
    _g._agents.clear()
    print(f'Reloaded {len(mods)} modules')

# Uncomment to reload:
# reload_all()

## 1. Config & LangSmith Tracing Setup

In [ ]:
from config.settings import get_settings

settings = get_settings()
print(f"Provider: {settings.resolved_provider}")
print(f"Model:    {settings.resolved_model}")
print(f"Sub-agent model: {settings.sub_agent_model or '(same as main)'}")

# Enable LangSmith tracing
settings.setup_langsmith()
print(f"\nLangSmith tracing: {os.environ.get('LANGCHAIN_TRACING_V2', 'false')}")
print(f"LangSmith project: {os.environ.get('LANGCHAIN_PROJECT', 'N/A')}")
print(f"LangSmith endpoint: {os.environ.get('LANGCHAIN_ENDPOINT', 'N/A')}")

In [ ]:
# Trace: LLM ping (check LangSmith for this trace)
llm = settings.get_llm()
resp = llm.invoke("Say 'hello' in one word.")
print(f"LLM response: {resp.content}")
print("PASS — check LangSmith for the trace of this call")

## 2. Data Layer: Catalog & Cache

In [ ]:
from data.loaders import load_catalog, list_datasets, get_cache, load_dataset, get_celltype_column
from data.metadata_store import load_metadata, find_similar_genes, get_gene_list, get_celltypes

catalog = load_catalog()
print(f"Catalog: {len(catalog)} datasets — {list(catalog.keys())}")
assert "mouse_brain_seqfish" in catalog
assert "mouse_brain_visium" in catalog
print(f"seqFISH celltype column: {get_celltype_column('mouse_brain_seqfish')}")
print(f"Visium celltype column:  {get_celltype_column('mouse_brain_visium')}")

# Check metadata store
meta_seq = load_metadata("mouse_brain_seqfish")
meta_vis = load_metadata("mouse_brain_visium")
print(f"\nMetadata store:")
print(f"  seqFISH: {meta_seq['n_genes']} genes, {len(meta_seq['celltypes'])} celltypes")
print(f"  Visium:  {meta_vis['n_genes']} genes, {len(meta_vis['celltypes'])} celltypes")
print("PASS: Catalog and metadata loaded")

In [ ]:
# Trace: Metadata store — fuzzy gene matching
print("Fuzzy gene matching tests:")

# Exact case-insensitive
r1 = find_similar_genes("mouse_brain_seqfish", "sox2")
print(f"  'sox2' → {r1}")
assert "Sox2" in r1

# Fuzzy match (typo)
r2 = find_similar_genes("mouse_brain_seqfish", "Sox3")
print(f"  'Sox3' (typo) → {r2}")

# Substring match
r3 = find_similar_genes("mouse_brain_seqfish", "Sox")
print(f"  'Sox' (partial) → {r3}")
assert any("Sox" in g for g in r3)

# Non-existent gene — should return suggestions
r4 = find_similar_genes("mouse_brain_seqfish", "BRCA1")
print(f"  'BRCA1' (not in dataset) → {r4}")

# Visium dataset fuzzy match
r5 = find_similar_genes("mouse_brain_visium", "gapdh")
print(f"  Visium 'gapdh' → {r5}")

print("PASS: Fuzzy gene matching working across datasets")

In [ ]:
# Load mouse brain seqFISH dataset (from h5ad, ~31 MB)
print("Loading mouse_brain_seqfish...")
adata = load_dataset("mouse_brain_seqfish")
print(f"Shape:       {adata.shape[0]} cells × {adata.shape[1]} genes")
print(f"Spatial:     {'spatial' in adata.obsm}")
print(f"Annotations: {list(adata.obs.columns)}")

assert "Sox2" in adata.var_names
print("\nPASS: Sox2 found in dataset")

cache = get_cache()
print(f"Cache: {cache.loaded_ids()}")
assert cache.is_loaded("mouse_brain_seqfish")
print("PASS: Dataset cached in memory")

## 3. PlotStore — Plots Stay Out of Context

In [ ]:
import matplotlib.pyplot as plt
from tools.base import fig_to_plot_id, get_plot_base64, clear_plot_store, tool_result

# Create a test plot
clear_plot_store()
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot([1, 2, 3], [1, 4, 9])
ax.set_title("Test plot")

pid = fig_to_plot_id(fig)
print(f"Plot ID:     {pid}  (8 chars, not base64!)")
print(f"Base64 size: {len(get_plot_base64(pid)):,} chars  (stored in PlotStore, NOT in LLM context)")

# Verify tool_result uses plot_id, not plot_base64
r = json.loads(tool_result(success=True, message="test", plot_id=pid))
assert "plot_id" in r
assert "plot_base64" not in r
print(f"\ntool_result keys: {list(r.keys())}")
print("PASS: Plots stored by ID — base64 never enters LLM context")

## 4. Trace: Individual Tool Calls

In [ ]:
from tools.dataset_tools import search_datasets, load_and_summarize_dataset, validate_gene

# Trace: search_datasets
r = json.loads(search_datasets.invoke({"query": "mouse brain seqfish"}))
print(f"search_datasets: success={r['success']}")
print(f"  matches: {r['data']['matches']}")
assert r["success"] and "mouse_brain_seqfish" in r["data"]["matches"]
print("PASS")

In [ ]:
# Trace: load_and_summarize_dataset
r = json.loads(load_and_summarize_dataset.invoke({"dataset_id": "mouse_brain_seqfish"}))
print(f"load_and_summarize: success={r['success']}")
print(f"  message: {r['message'][:120]}")
print(f"  data keys: {list(r['data'].keys())}")
# Verify compact — annotations capped at 10
assert len(r["data"]["annotations"]) <= 10, "FAIL: annotations not capped!"
print(f"  annotations count: {len(r['data']['annotations'])} (capped at 10)")
assert r["data"]["dataset_id"] == "mouse_brain_seqfish"
print("PASS")

In [ ]:
# Trace: validate_gene (exact + case-insensitive + fuzzy suggestions)
r1 = json.loads(validate_gene.invoke({"dataset_id": "mouse_brain_seqfish", "gene_name": "Sox2"}))
r2 = json.loads(validate_gene.invoke({"dataset_id": "mouse_brain_seqfish", "gene_name": "sox2"}))
print(f"validate_gene('Sox2'): {r1['data']['gene']}")
print(f"validate_gene('sox2'): {r2['data']['gene']}")
assert r1["success"] and r2["success"]
assert r1["data"]["gene"] == r2["data"]["gene"] == "Sox2"
print("PASS: Gene validation works (exact + case-insensitive)")

# Test fuzzy suggestion on non-existent gene
r3 = json.loads(validate_gene.invoke({"dataset_id": "mouse_brain_seqfish", "gene_name": "Snap25"}))
print(f"\nvalidate_gene('Snap25'): success={r3['success']}")
print(f"  message: {r3['message']}")
assert not r3["success"]
# Should include suggestions from fuzzy matching
if "suggestions" in r3.get("data", {}):
    print(f"  suggestions: {r3['data']['suggestions']}")
print("PASS: Non-existent gene returns helpful error with suggestions")

In [ ]:
import base64
from IPython.display import Image, display
from tools.expression_tools import get_gene_expression_spatial, plot_celltype_spatial, gene_expression_by_celltype

clear_plot_store()

# Trace: get_gene_expression_spatial
r = json.loads(get_gene_expression_spatial.invoke({
    "dataset_id": "mouse_brain_seqfish", "gene": "Sox2"
}))
print(f"expression: success={r['success']}")
print(f"  message: {r['message']}")
print(f"  data: {r['data']}")
print(f"  plot_id: {r.get('plot_id')}")

# KEY CHECK: no base64 in tool result
assert "plot_base64" not in r, "FAIL: base64 leaked into tool result!"
assert "plot_id" in r, "FAIL: plot_id missing!"
print("\nPASS: Tool result is compact (plot_id only, no base64)")
print(f"  Result size: {len(json.dumps(r)):,} chars")
print(f"  Plot base64 size: {len(get_plot_base64(r['plot_id'])):,} chars (in PlotStore, not in result)")

# Display the plot from PlotStore
display(Image(data=base64.b64decode(get_plot_base64(r["plot_id"]))))
print("Plot displayed from PlotStore")

# Trace: plot_celltype_spatial (new tool!)
print("\n--- plot_celltype_spatial ---")
r2 = json.loads(plot_celltype_spatial.invoke({"dataset_id": "mouse_brain_seqfish"}))
print(f"celltype spatial: success={r2['success']}")
print(f"  message: {r2['message']}")
if r2.get("plot_id"):
    display(Image(data=base64.b64decode(get_plot_base64(r2["plot_id"]))))

# Trace: gene_expression_by_celltype (new tool!)
print("\n--- gene_expression_by_celltype ---")
r3 = json.loads(gene_expression_by_celltype.invoke({"dataset_id": "mouse_brain_seqfish", "gene": "Sox2"}))
print(f"gene by celltype: success={r3['success']}")
print(f"  message: {r3['message']}")
if r3.get("plot_id"):
    display(Image(data=base64.b64decode(get_plot_base64(r3["plot_id"]))))

print("\nPASS: All expression tools working")

## 5. Trace: bind_tools Sub-Agent (Dataset Finder)

In [ ]:
from langchain_core.messages import HumanMessage
from agents.sub_agents import create_dataset_finder_agent
from tools.base import clear_plot_store

clear_plot_store()
agent_fn = create_dataset_finder_agent()
print("Created dataset_finder agent (llm.bind_tools)")
print("This agent uses bind_tools for multi-step ReAct with proper ToolMessage + tool_call_id.")
print("Pydantic validation on args is handled by LangChain's @tool decorator.\n")

# Run the agent
result = agent_fn([HumanMessage(content="Sox2 expression in Mouse Brain seqFISH")])

print(f"summary: {result['summary'][:200]}")
print(f"tool_summaries: {result['tool_summaries']}")
print(f"plot_ids: {result['plot_ids']}")
print(f"detected_dataset_id: {result['detected_dataset_id']}")

assert result["detected_dataset_id"] == "mouse_brain_seqfish"
print("\nPASS: dataset_finder agent found and loaded the dataset")

## 6. Trace: bind_tools Sub-Agent (Exploratory)

In [ ]:
from agents.sub_agents import create_exploratory_agent

clear_plot_store()
agent_fn = create_exploratory_agent()
print("Created exploratory agent (llm.bind_tools)\n")

# Pass dataset context like the graph does
result = agent_fn([
    HumanMessage(content="[Dataset: mouse_brain_seqfish] Show Sox2 expression")
])

print(f"summary: {result['summary'][:200]}")
print(f"tool_summaries: {result['tool_summaries']}")
print(f"plot_ids: {result['plot_ids']}")

# KEY CHECK: plots generated but not in summary text
assert len(result["plot_ids"]) > 0, "FAIL: No plots generated"
for pid in result["plot_ids"]:
    b64 = get_plot_base64(pid)
    assert b64 is not None
    print(f"  Plot {pid}: {len(b64):,} chars in PlotStore")

# Display
for pid in result["plot_ids"]:
    display(Image(data=base64.b64decode(get_plot_base64(pid))))

# Verify compact summaries
for s in result["tool_summaries"]:
    assert len(s) <= 300, f"FAIL: Summary too long ({len(s)} chars)"
print("\nPASS: Exploratory agent generated plot, compact summaries")

## 7. Trace: Supervisor Routing

In [ ]:
from agents.supervisor import create_supervisor_node

supervisor = create_supervisor_node()

# Scenario 1: Fresh — should route to dataset_finder
state1 = {
    "messages": [HumanMessage(content="Sox2 expression in Mouse Brain seqFISH")],
    "tool_summaries": [], "plot_ids": [], "visited_agents": [],
    "supervisor_turns": 0, "error_count": 0,
    "active_dataset_id": None, "next_agent": None,
}
r1 = supervisor(state1)
print(f"Scenario 1 (fresh):              next_agent = {r1['next_agent']}")
assert r1["next_agent"] == "dataset_finder"

# Scenario 2: Dataset loaded, df visited — fast path to exploratory (NO LLM call!)
state2 = {
    "messages": [HumanMessage(content="Sox2 expression in Mouse Brain seqFISH")],
    "tool_summaries": ["Loaded mouse_brain_seqfish"], "plot_ids": [],
    "visited_agents": ["dataset_finder"],
    "supervisor_turns": 1, "error_count": 0,
    "active_dataset_id": "mouse_brain_seqfish", "next_agent": None,
}
r2 = supervisor(state2)
print(f"Scenario 2 (ds loaded, df done):  next_agent = {r2['next_agent']} (pre-LLM fast path)")
assert r2["next_agent"] == "exploratory"

# Scenario 3: Both visited — FINISH
state3 = {
    "messages": [HumanMessage(content="Sox2 expression in Mouse Brain seqFISH")],
    "tool_summaries": ["Loaded", "Sox2 plotted"], "plot_ids": ["abc"],
    "visited_agents": ["dataset_finder", "exploratory"],
    "supervisor_turns": 2, "error_count": 0,
    "active_dataset_id": "mouse_brain_seqfish", "next_agent": None,
}
r3 = supervisor(state3)
print(f"Scenario 3 (both visited):        next_agent = {r3['next_agent']}")
assert r3["next_agent"] == "FINISH"

# Scenario 4: Max turns breaker
state4 = {**state1, "supervisor_turns": 4}
r4 = supervisor(state4)
print(f"Scenario 4 (max turns):           next_agent = {r4['next_agent']}")
assert r4["next_agent"] == "FINISH"

print("\nPASS: All supervisor routing scenarios correct")

In [ ]:
# Trace: route_from_supervisor (graph-level router)
from graph import route_from_supervisor

tests = [
    ({"next_agent": "dataset_finder"},   "dataset_finder"),
    ({"next_agent": "exploratory"},      "exploratory"),
    ({"next_agent": "spatial_stats"},    "spatial_stats"),
    ({"next_agent": "neighborhood"},     "neighborhood"),
    ({"next_agent": "FINISH"},           "synthesizer"),
    ({"next_agent": None},               "synthesizer"),
    ({"next_agent": "hallucinated"},     "synthesizer"),
]
for state, expected in tests:
    actual = route_from_supervisor(state)
    status = "PASS" if actual == expected else "FAIL"
    print(f"  {status}: route({str(state['next_agent']):20s}) -> {actual}")
    assert actual == expected

print("PASS: All routing edges correct")

## 8. Trace: Build & Inspect Graph

In [ ]:
from graph import build_graph

# Build without checkpointer for stateless notebook testing
graph = build_graph()
print("Graph compiled (stateless for notebook testing)")
print("Note: make_graph() for LangGraph Studio uses MemorySaver for multi-turn")
print()

# Visualize
try:
    png = graph.get_graph().draw_mermaid_png()
    display(Image(data=png))
except Exception as e:
    print(f"Graph viz unavailable ({e})")
    print("Flow: reset → supervisor → dataset_finder → supervisor → exploratory → supervisor → synthesizer → END")

## 9. Trace: Full Graph — Stream Every Step

This is the **key tracing cell**. It streams the graph and logs every node update.

Watch for:
- `tool_summaries` should be short strings (not huge dicts)
- `plot_ids` should be 8-char IDs (not base64)
- No `plot_base64` or `tool_results` keys anywhere
- Total steps should be ≤ 8 (reset→sup→df→sup→exp→sup→synth, with reset node)

In [ ]:
from tools.base import clear_plot_store, get_plot_base64

clear_plot_store()

initial_state = {
    "messages": [HumanMessage(content="Sox2 expression in Mouse Brain seqFISH")],
    "active_dataset_id": None,
    "tool_summaries": [],
    "plot_ids": [],
    "visited_agents": [],
    "supervisor_turns": 0,
    "error_count": 0,
}

print("=" * 70)
print("STREAMING: 'Sox2 expression in Mouse Brain seqFISH'")
print("=" * 70)

step = 0
all_context_sizes = []

for event in graph.stream(initial_state, stream_mode="updates"):
    step += 1
    for node_name, update in event.items():
        print(f"\n{'─' * 50}")
        print(f"Step {step}: {node_name}")
        print(f"{'─' * 50}")

        if "next_agent" in update:
            print(f"  next_agent:       {update['next_agent']}")
        if "supervisor_turns" in update:
            print(f"  supervisor_turns: {update['supervisor_turns']}")
        if "visited_agents" in update:
            print(f"  visited_agents:   {update['visited_agents']}")
        if "active_dataset_id" in update:
            print(f"  active_dataset:   {update['active_dataset_id']}")

        # Trace tool summaries — should be compact
        if "tool_summaries" in update:
            for s in update["tool_summaries"]:
                size = len(str(s))
                all_context_sizes.append(size)
                print(f"  tool_summary ({size} chars): {str(s)[:120]}")

        # Trace plot IDs — should be short
        if "plot_ids" in update:
            for pid in update["plot_ids"]:
                b64_size = len(get_plot_base64(pid) or "")
                print(f"  plot_id: {pid}  (base64={b64_size:,} chars in PlotStore)")

        # Trace messages — should be short summaries
        if "messages" in update:
            for msg in update["messages"]:
                mtype = getattr(msg, 'type', 'unknown')
                content = str(msg.content)[:150] if hasattr(msg, 'content') else ''
                print(f"  message ({mtype}): {content}")

        # BLOAT CHECK: flag anything suspicious
        update_str = str(update)
        if "iVBOR" in update_str:
            print(f"  WARNING: base64 PNG detected in state update!")
        if len(update_str) > 5000:
            print(f"  WARNING: Large state update ({len(update_str):,} chars)")

print(f"\n{'=' * 70}")
print(f"Completed in {step} steps")
print(f"Expected: ≤8 steps (reset→sup→df→sup→exp→sup→synth)")
print(f"{'=' * 70}")

In [ ]:
# Context bloat audit
print("Context Size Audit:")
print(f"  Tool summaries generated: {len(all_context_sizes)}")
if all_context_sizes:
    print(f"  Max summary size: {max(all_context_sizes)} chars")
    print(f"  Avg summary size: {sum(all_context_sizes) / len(all_context_sizes):.0f} chars")
    assert max(all_context_sizes) <= 300, f"FAIL: Summary too large ({max(all_context_sizes)} chars)!"
    print("  PASS: All summaries ≤ 300 chars")

assert step <= 8, f"FAIL: Too many steps ({step}). Likely looping."
print(f"  PASS: {step} steps (no looping)")

## 10. Trace: Full End-to-End via `chat()`

The `chat()` function is the public API. It:
1. Uses MemorySaver checkpointer for multi-turn conversation
2. The `reset` node clears per-invocation state each turn
3. Extracts response text (handles multimodal content)
4. Retrieves plot base64 from PlotStore using IDs
5. Returns `active_dataset_id` for session state

In [ ]:
from graph import chat

# First call: load + analyze
print("Call 1: 'Sox2 expression in Mouse Brain seqFISH'")
print("-" * 50)
result = chat("Sox2 expression in Mouse Brain seqFISH", thread_id="notebook-test")

print(f"Response ({len(result['response'])} chars):")
print(result["response"][:400])
print(f"\nPlots: {len(result['plots'])}")
print(f"Active dataset: {result['active_dataset_id']}")

# KEY CHECKS
assert len(result["response"]) > 50, "FAIL: Response too short"
assert result["active_dataset_id"] is not None, "FAIL: No dataset ID returned"

# Display plots
for i, p in enumerate(result["plots"]):
    print(f"\nPlot {i+1} ({len(p):,} chars):")
    display(Image(data=base64.b64decode(p)))

print("\nPASS: Full pipeline complete")

In [ ]:
# Second call: follow-up on the SAME thread (multi-turn conversation!)
# The reset node clears per-invocation state, but messages + active_dataset_id persist
print("Call 2: Follow-up 'Show cell types spatially' (same thread)")
print("-" * 50)

result2 = chat("Show cell types spatially", thread_id="notebook-test")

print(f"Response ({len(result2['response'])} chars):")
print(result2["response"][:300])
print(f"\nPlots: {len(result2['plots'])}")
print(f"Active dataset: {result2['active_dataset_id']}")

for i, p in enumerate(result2["plots"]):
    display(Image(data=base64.b64decode(p)))

# The agent should have used the previously loaded dataset
assert result2["active_dataset_id"] is not None
print("\nPASS: Multi-turn conversation works (dataset persisted across turns)")

## 11. Trace: Loop Prevention Verification

In [ ]:
clear_plot_store()
node_counts = {}

for event in graph.stream(
    {"messages": [HumanMessage(content="Sox2 expression in Mouse Brain seqFISH")],
     "active_dataset_id": None,
     "tool_summaries": [], "plot_ids": [],
     "visited_agents": [], "supervisor_turns": 0, "error_count": 0},
    stream_mode="updates",
):
    for node_name in event:
        node_counts[node_name] = node_counts.get(node_name, 0) + 1

print(f"Node visit counts: {node_counts}")

# reset node runs exactly once
reset_count = node_counts.get("reset", 0)
assert reset_count == 1, f"FAIL: reset ran {reset_count} times"
print(f"  reset:          {reset_count} (exactly 1) — PASS")

df_count = node_counts.get("dataset_finder", 0)
assert df_count <= 1, f"FAIL: dataset_finder ran {df_count} times"
print(f"  dataset_finder: {df_count} (max 1) — PASS")

exp_count = node_counts.get("exploratory", 0)
assert exp_count <= 1, f"FAIL: exploratory ran {exp_count} times"
print(f"  exploratory:    {exp_count} (max 1) — PASS")

sup_count = node_counts.get("supervisor", 0)
assert sup_count <= 4, f"FAIL: supervisor ran {sup_count} times"
print(f"  supervisor:     {sup_count} (max 4) — PASS")

syn_count = node_counts.get("synthesizer", 0)
assert syn_count == 1, f"FAIL: synthesizer ran {syn_count} times"
print(f"  synthesizer:    {syn_count} (exactly 1) — PASS")

total = sum(node_counts.values())
print(f"\nTotal: {total} node executions (expected ≤ 8, including reset)")
assert total <= 8
print("PASS: No looping")

## 12. LangSmith Trace Links

If LangSmith tracing is enabled, view your traces at:

**https://smith.langchain.com**

Each cell above that called an LLM or tool should have a corresponding trace showing:
- Input/output messages
- Token usage
- Latency per step
- Tool call validation (trustcall retries, if any)
- Graph node transitions

In [ ]:
project = os.environ.get("LANGCHAIN_PROJECT", "spatialchat")
tracing = os.environ.get("LANGCHAIN_TRACING_V2", "false")

if tracing == "true":
    url = f"https://smith.langchain.com/o/default/projects/p/{project}"
    print(f"LangSmith traces available at:")
    print(f"  {url}")
    print(f"\nLook for runs from project '{project}'")
    print("Each graph invocation shows the full supervisor → sub-agent → synthesizer flow.")
else:
    print("LangSmith tracing is OFF.")
    print("To enable, add to your .env:")
    print("  LANGCHAIN_TRACING_V2=true")
    print("  LANGCHAIN_API_KEY=lsv2_pt_...")
    print("  LANGCHAIN_PROJECT=spatialchat")

---

## Summary of What Was Traced

| Step | Component | What's Traced |
|------|-----------|---------------|
| 1 | Config & LLM | Provider, model, LangSmith connection |
| 2 | Data layer | Catalog load (2 datasets), universal h5ad loader, cache, **metadata store** |
| 2b | Metadata store | Fuzzy gene matching (case-insensitive, edit-distance, substring, prefix) |
| 3 | PlotStore | `fig_to_plot_id()` — plots stored by ID, not base64 |
| 4 | Tool calls | `search_datasets`, `load_and_summarize`, `validate_gene` (with fuzzy suggestions), `expression`, **`plot_celltype_spatial`**, **`gene_expression_by_celltype`** |
| 5 | Sub-agent (DF) | `bind_tools` ReAct loop → proper ToolMessage + tool_call_id |
| 6 | Sub-agent (Exp) | `bind_tools` ReAct loop → gene expression with plot |
| 7 | Supervisor | 4 routing scenarios: fresh, fast-path, finish, max-turns |
| 8 | Graph build | Compiled graph with reset node (optional MemorySaver) |
| 9 | Full stream | Every node update with context bloat audit |
| 10 | `chat()` API | End-to-end + multi-turn conversation (thread_id) |
| 11 | Loop check | Node visit counts verify no re-routing |
| 12 | LangSmith | Link to traces dashboard |

### Datasets Available
| Dataset | Cells | Genes | Technology | Celltype Column |
|---------|-------|-------|------------|-----------------|
| `mouse_brain_seqfish` | 19,416 | 351 | seqFISH | `celltype_mapped_refined` |
| `mouse_brain_visium` | 2,688 | 18,078 | 10x Visium | `cluster` |